# Unit 7 — SoccerTwos: Multi-agent RL (2 vs 2 football) — **run locally**

**Pass criterion:** threshold −100 → **train and push the model.** The pushed model automatically enters the **AI vs AI tournament** and receives an ELO rating.

**Why not Colab?** 5M training steps take **5–8 hours**, longer than a free Colab session lasts. ML-Agents trains on the CPU, so a local machine works well.

---
## 0. One-time setup (in **Anaconda Prompt**)
Requires Anaconda / Miniconda. Then:

```bat
conda create -n rl python=3.10.12 -y
conda activate rl
pip install notebook
mkdir C:\Dev\deep-rl-course
cd C:\Dev\deep-rl-course
jupyter notebook
```
Put this notebook (`unit7_soccertwos_LOCAL.ipynb`) into `C:\Dev\deep-rl-course`, open it in Jupyter and run the cells in order.

> ML-Agents **only supports Python 3.10.x**, hence the separate `rl` environment. It does not affect other projects.

## 1. Correct environment?

In [ ]:
import sys, platform
print(sys.version)
assert sys.version.startswith("3.10"), "Open this notebook with the Jupyter of the 'rl' conda environment (conda activate rl)!"
print(platform.system())

## 2. Install ML-Agents (~5 min)
The `PATH` line makes `!mlagents-learn` and other shell commands resolve inside the **rl environment** instead of Jupyter's base environment.
ML-Agents is installed straight from PyPI (no git clone → no Windows long-path issues).

Two pins avoid real-world breakages:
- `setuptools<81` — newer setuptools removed `pkg_resources`, which TensorBoard and ML-Agents still import.
- `torch==2.2.2` (CPU) — recent PyTorch switched `torch.onnx.export` to a new exporter that needs `onnxscript`; training crashed at the first checkpoint (500k steps) until PyTorch was pinned.

If you restart the notebook, **re-run cells 1 and 2** first.

In [ ]:
import os, sys
BASE = r"C:\Dev\deep-rl-course"
os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)
# Make "!" shell commands (mlagents-learn, mlagents-push-to-hf) use THIS kernel's environment (rl)
os.environ["PATH"] = os.pathsep.join([os.path.join(sys.prefix, "Scripts"), sys.prefix, os.environ["PATH"]])
# ML-Agents from PyPI (no git clone needed; avoids Windows long-path errors)
%pip install -q mlagents==1.1.0 gdown tensorboard "setuptools<81"
# Newest PyTorch breaks ML-Agents ONNX export -> pin the tested version (CPU build)
%pip install -q "torch==2.2.2" "numpy<2" --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username

## 3. Download the SoccerTwos environment
The course's **Windows** build is downloaded from Google Drive and extracted.

In [ ]:
import glob, zipfile, gdown
os.chdir(BASE)
os.makedirs("training-envs-executables", exist_ok=True)
zip_path = "training-envs-executables/SoccerTwos.zip"
if not os.path.exists(zip_path):
    gdown.download(id="1sqFxbEdTMubjVktnV4C6ICjp89wLhUcP", output=zip_path, quiet=False)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("training-envs-executables")
EXE = [p for p in glob.glob("training-envs-executables/**/SoccerTwos.exe", recursive=True)][0]
EXE = os.path.abspath(EXE)
print("Game executable:", EXE)

## 4. Training config
- `trainer_type: poca` → **MA-POCA**: multi-agent PPO. The two teammates are trained with a **shared centralised critic**, so a player's contribution is credited to the team even after it is removed.
- `self_play` → the agent plays against **past versions of itself**:
  - `save_steps: 50000` → a snapshot is added to the opponent pool every 50k steps.
  - `window: 10` → the pool keeps the last 10 snapshots.
  - `play_against_latest_model_ratio: 0.5` → half of the games are against the latest opponent.
  - `swap_steps`, `team_change` → how often the opponent and the learning team are switched.
  - `initial_elo: 1200` → starting ELO rating.
- `max_steps: 5000000` → 5M steps (~5–8 hours).

In [ ]:
%%writefile SoccerTwos.yaml
behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    max_steps: 5000000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000
      team_change: 200000
      swap_steps: 2000
      window: 10
      play_against_latest_model_ratio: 0.5
      initial_elo: 1200.0

## 5. Live training dashboard: TensorBoard
Refresh with ⟳ during training. Most important chart: **`Self-play/ELO`** → if it rises, the agent beats its past versions, i.e. it is improving.

> On Windows, Jupyter may only show the output of `!` shell commands when the command finishes, so TensorBoard is the reliable way to follow progress.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results

## 6. Training (5–8 hours)
- Disable sleep mode (Settings → System → Power → Sleep: Never) and keep the laptop plugged in.
- If training stops: replace `--force` with `--resume` in the cell below and run it again → training resumes from the latest checkpoint.
- The browser tab can be closed while this cell runs; training continues as long as Jupyter (the Anaconda Prompt window) stays open.

In [ ]:
!mlagents-learn SoccerTwos.yaml --env="{EXE}" --run-id="SoccerTwos" --no-graphics --force

## 7. Watch 🎮 (after training)
The cell below opens the game **with graphics** and shows the trained model playing (no learning). Close the game window or stop the cell (■) when you are done.

In [ ]:
!mlagents-learn SoccerTwos.yaml --env="{EXE}" --run-id="SoccerTwos" --resume --inference --time-scale=1

## 8. Push to the Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
!mlagents-push-to-hf --run-id="SoccerTwos" --local-dir="./results/SoccerTwos" --repo-id="{HF_USERNAME}/poca-SoccerTwos" --commit-message="SoccerTwos MA-POCA self-play agent"

## 9. Check and tournament
- The cell below runs the certification query.
- Tournament: https://huggingface.co/spaces/huggingface-projects/AIvsAI-SoccerTwos (it can take a few hours before the model plays matches and gets an ELO)
- Watch in the browser: https://huggingface.co/spaces/unity/ML-Agents-SoccerTwos

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("ML-Agents-SoccerTwos", "ml-agents", -100)